# Includes

In [52]:
import pandas as pd
import numpy as np
import mne
import mne_nirs
import h5py
import shutil
import plotly.express as px

from pathlib import Path
from mne.preprocessing.nirs import source_detector_distances, short_channels, optical_density, beer_lambert_law

root = Path.home() / "fnirs-representation-learning"
rs_data_dir = root / "snirf_dataset_2"

In [53]:
clean_file_rows = []

for subject_dir in sorted(rs_data_dir.glob("Subj*")):
    if not subject_dir.is_dir():
        continue

    resting_file = subject_dir / "resting.snirf"
    clean_file = subject_dir / "resting_clean.snirf"

    if not clean_file.exists():
        shutil.copy2(resting_file, clean_file)

        with h5py.File(clean_file, "r+") as f:
            if "stim1" in f["nirs"]:
                del f["nirs"]["stim1"]

        print("created:", clean_file)
        
    else:
        print("already exists:", clean_file)

    clean_file_rows.append({
        "subject": subject_dir.name,
        "clean_file": clean_file,
    })

clean_files_df = pd.DataFrame(clean_file_rows)
clean_files_df

already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj101/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj91/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj92/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learnin

,subject,clean_file
0,Subj100,/home/asunkari/fnirs-representation-learning/s...
1,Subj101,/home/asunkari/fnirs-representation-learning/s...
2,Subj102,/home/asunkari/fnirs-representation-learning/s...
3,Subj103,/home/asunkari/fnirs-representation-learning/s...
4,Subj104,/home/asunkari/fnirs-representation-learning/s...
5,Subj86,/home/asunkari/fnirs-representation-learning/s...
6,Subj91,/home/asunkari/fnirs-representation-learning/s...
7,Subj92,/home/asunkari/fnirs-representation-learning/s...
8,Subj94,/home/asunkari/fnirs-representation-learning/s...
9,Subj95,/home/asunkari/fnirs-representation-learning/s...


In [54]:
subject_summary_rows = []
pair_rows = []

for row in clean_files_df.itertuples(index=False):
    subject = row.subject
    clean_file = row.clean_file

    raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)

    # pick all fNIRS channels, then keep only CW amplitude channels
    picks_fnirs = mne.pick_types(raw_rest.info, fnirs=True)
    channel_types = np.array(raw_rest.get_channel_types())
    picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]

    # distances and SS/LS masks
    dists = source_detector_distances(raw_rest.info, picks=picks_cw)

    ss_mask_all = short_channels(raw_rest.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_cw]

    ls_mask = dists >= 0.025

    # channel-level names
    cw_names = np.array(raw_rest.ch_names)[picks_cw]
    pair_names = np.array([name.split(" ")[0] for name in cw_names])

    # pair-level table for this subject
    pair_table = pd.DataFrame({
        "subject": subject,
        "channel_name": cw_names,
        "pair_name": pair_names,
        "distance_m": dists,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    pair_summary = (
        pair_table.groupby(["subject", "pair_name"], as_index=False)
        .agg(
            distance_m=("distance_m", "first"),
            is_ss=("is_ss", "first"),
            is_ls=("is_ls", "first"),
        )
    )

    pair_summary["group"] = np.select(
        [pair_summary["is_ss"], pair_summary["is_ls"]],
        ["SS", "LS"],
        default="MID"
    )

    pair_rows.append(pair_summary)

    group_counts = pair_summary["group"].value_counts()

    subject_summary_rows.append({
        "subject": subject,
        "file": str(clean_file),
        "sfreq": raw_rest.info["sfreq"],
        "duration_s": raw_rest.times[-1],
        "n_fnirs_channels": len(picks_fnirs),
        "n_cw_channels": len(picks_cw),
        "distance_min_m": float(dists.min()),
        "distance_max_m": float(dists.max()),
        "n_ss_channels": int(ss_mask.sum()),
        "n_ls_channels": int(ls_mask.sum()),
        "n_pairs_total": len(pair_summary),
        "n_pairs_ss": int(group_counts.get("SS", 0)),
        "n_pairs_ls": int(group_counts.get("LS", 0)),
        "n_pairs_mid": int(group_counts.get("MID", 0)),
    })

subject_summary_df = pd.DataFrame(subject_summary_rows)
pair_summary_df = pd.concat(pair_rows, ignore_index=True)

subject_summary_df

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj101/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj91/resting_clean.snirf


...

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj96/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj97/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj98/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data o

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj99/resting_clean.snirf


/tmp/ipykernel_384858/3600149802.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



,subject,file,sfreq,duration_s,n_fnirs_channels,n_cw_channels,distance_min_m,distance_max_m,n_ss_channels,n_ls_channels,n_pairs_total,n_pairs_ss,n_pairs_ls,n_pairs_mid
0,Subj100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,0.008,0.030463,16,96,56,8,48,0
1,Subj101,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,0.008,0.030463,16,96,56,8,48,0
2,Subj102,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,0.008,0.030463,16,96,56,8,48,0
3,Subj103,/home/asunkari/fnirs-representation-learning/s...,50.0,716.98,112,112,0.008,0.030463,16,96,56,8,48,0
4,Subj104,/home/asunkari/fnirs-representation-learning/s...,50.0,669.98,112,112,0.008,0.030463,16,96,56,8,48,0
5,Subj86,/home/asunkari/fnirs-representation-learning/s...,50.0,590.98,112,112,0.008,0.030463,16,96,56,8,48,0
6,Subj91,/home/asunkari/fnirs-representation-learning/s...,50.0,641.98,112,112,0.008,0.030463,16,96,56,8,48,0
7,Subj92,/home/asunkari/fnirs-representation-learning/s...,50.0,588.98,112,112,0.008,0.030463,16,96,56,8,48,0
8,Subj94,/home/asunkari/fnirs-representation-learning/s...,50.0,692.98,112,112,0.008,0.030463,16,96,56,8,48,0
9,Subj95,/home/asunkari/fnirs-representation-learning/s...,50.0,765.98,112,112,0.008,0.030463,16,96,56,8,48,0


# Inspect semisynthetic HRF files and event annotations

In [55]:
output_dir = root / "outputs"
output_tables_dir = output_dir / "tables"
output_figures_dir = output_dir / "figures"

output_tables_dir.mkdir(parents=True, exist_ok=True)
output_figures_dir.mkdir(parents=True, exist_ok=True)

file_labels = [
    ("clean", "resting_clean.snirf"),
    ("hrf_20", "resting_hrf_20.snirf"),
    ("hrf_50", "resting_hrf_50.snirf"),
    ("hrf_100", "resting_hrf_100.snirf"),
]

In [56]:
def get_cw_channel_indices(raw_snirf):
    picks_fnirs = mne.pick_types(raw_snirf.info, fnirs=True)
    channel_types = np.array(raw_snirf.get_channel_types())
    picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]
    return picks_cw


def build_channel_table(raw_snirf, subject_name, file_label):
    picks_cw = get_cw_channel_indices(raw_snirf)
    cw_names = np.array(raw_snirf.ch_names)[picks_cw]

    dists = source_detector_distances(raw_snirf.info, picks=picks_cw)

    ss_mask_all = short_channels(raw_snirf.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_cw]

    ls_mask = dists >= 0.025
    pair_names = np.array([name.split(" ")[0] for name in cw_names])

    channel_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": cw_names,
        "pair_name": pair_names,
        "distance_m": dists,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    channel_table["group"] = np.select(
        [channel_table["is_ss"], channel_table["is_ls"]],
        ["SS", "LS"],
        default="MID",
    )

    return channel_table


def build_event_table(raw_snirf, subject_name, file_label):
    annotations = raw_snirf.annotations

    event_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "onset_s": annotations.onset,
        "duration_s": annotations.duration,
        "description": annotations.description,
    })

    return event_table

In [57]:
subject_name = "Subj100"
subject_dir = rs_data_dir / subject_name

subject_file_rows = []
subject_event_tables = []
subject_channel_tables = []

for file_label, file_name in file_labels:
    file_path = subject_dir / file_name

    if not file_path.exists():
        print("missing:", file_path)
        continue

    raw_snirf = mne.io.read_raw_snirf(file_path, preload=False, verbose=False)
    channel_table = build_channel_table(raw_snirf, subject_name, file_label)
    event_table = build_event_table(raw_snirf, subject_name, file_label)

    subject_file_rows.append({
        "subject": subject_name,
        "file_label": file_label,
        "file_path": str(file_path),
        "sfreq": raw_snirf.info["sfreq"],
        "duration_s": raw_snirf.times[-1],
        "n_channels_total": len(raw_snirf.ch_names),
        "n_cw_channels": len(channel_table),
        "n_pairs": channel_table["pair_name"].nunique(),
        "n_ss_pairs": int((channel_table.groupby("pair_name")["group"].first() == "SS").sum()),
        "n_ls_pairs": int((channel_table.groupby("pair_name")["group"].first() == "LS").sum()),
        "n_mid_pairs": int((channel_table.groupby("pair_name")["group"].first() == "MID").sum()),
        "n_annotations": len(raw_snirf.annotations),
        "annotation_descriptions": "|".join(sorted(set(raw_snirf.annotations.description))),
    })

    subject_channel_tables.append(channel_table)
    subject_event_tables.append(event_table)

subject_file_summary_df = pd.DataFrame(subject_file_rows)
subject_channel_summary_df = pd.concat(subject_channel_tables, ignore_index=True)
subject_event_summary_df = pd.concat(subject_event_tables, ignore_index=True)

subject_file_summary_df

/tmp/ipykernel_384858/2660192721.py:15: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2660192721.py:15: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2660192721.py:15: RuntimeWarning:

The dat

,subject,file_label,file_path,sfreq,duration_s,n_channels_total,n_cw_channels,n_pairs,n_ss_pairs,n_ls_pairs,n_mid_pairs,n_annotations,annotation_descriptions
0,Subj100,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,0,
1,Subj100,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
2,Subj100,hrf_50,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
3,Subj100,hrf_100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1


In [58]:
subject_event_summary_df

,subject,file_label,onset_s,duration_s,description
0,Subj100,hrf_20,1.22,1.0,1
1,Subj100,hrf_20,21.06,1.0,1
2,Subj100,hrf_20,41.42,1.0,1
3,Subj100,hrf_20,61.06,1.0,1
4,Subj100,hrf_20,82.66,1.0,1
...,...,...,...,...,...
103,Subj100,hrf_100,621.06,1.0,1
104,Subj100,hrf_100,641.42,1.0,1
105,Subj100,hrf_100,661.82,1.0,1
106,Subj100,hrf_100,680.22,1.0,1


In [59]:
annotation_figure = px.scatter(
    subject_event_summary_df,
    x="onset_s",
    y="description",
    color="file_label",
    hover_data=["duration_s"],
    title=f"{subject_name}: annotation overview",
)

annotation_figure.show()
annotation_figure.write_html(output_figures_dir / f"{subject_name.lower()}_annotation_overview.html")

In [60]:
all_file_rows = []
all_event_tables = []
all_channel_tables = []

for subject_dir in sorted(rs_data_dir.glob("Subj*")):
    if not subject_dir.is_dir():
        continue

    subject_name = subject_dir.name

    for file_label, file_name in file_labels:
        file_path = subject_dir / file_name

        if not file_path.exists():
            print("missing:", file_path)
            continue

        raw_snirf = mne.io.read_raw_snirf(file_path, preload=False, verbose=False)
        channel_table = build_channel_table(raw_snirf, subject_name, file_label)
        event_table = build_event_table(raw_snirf, subject_name, file_label)

        pair_groups = channel_table.groupby("pair_name")["group"].first()

        all_file_rows.append({
            "subject": subject_name,
            "file_label": file_label,
            "file_path": str(file_path),
            "sfreq": raw_snirf.info["sfreq"],
            "duration_s": raw_snirf.times[-1],
            "n_channels_total": len(raw_snirf.ch_names),
            "n_cw_channels": len(channel_table),
            "n_pairs": channel_table["pair_name"].nunique(),
            "n_ss_pairs": int((pair_groups == "SS").sum()),
            "n_ls_pairs": int((pair_groups == "LS").sum()),
            "n_mid_pairs": int((pair_groups == "MID").sum()),
            "n_annotations": len(raw_snirf.annotations),
            "annotation_descriptions": "|".join(sorted(set(raw_snirf.annotations.description))),
        })

        all_channel_tables.append(channel_table)
        all_event_tables.append(event_table)

all_file_summary_df = pd.DataFrame(all_file_rows)
all_channel_summary_df = pd.concat(all_channel_tables, ignore_index=True)
all_event_summary_df = pd.concat(all_event_tables, ignore_index=True)

/tmp/ipykernel_384858/2570191556.py:18: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2570191556.py:18: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/2570191556.py:18: RuntimeWarning:

The dat

In [61]:
all_file_summary_df

,subject,file_label,file_path,sfreq,duration_s,n_channels_total,n_cw_channels,n_pairs,n_ss_pairs,n_ls_pairs,n_mid_pairs,n_annotations,annotation_descriptions
0,Subj100,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,0,
1,Subj100,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
2,Subj100,hrf_50,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
3,Subj100,hrf_100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,56,8,48,0,36,1
4,Subj101,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,0,
5,Subj101,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1
6,Subj101,hrf_50,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1
7,Subj101,hrf_100,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1
8,Subj102,clean,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,0,
9,Subj102,hrf_20,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,56,8,48,0,36,1


In [62]:
all_file_summary_df.groupby("file_label")[["n_annotations", "n_ss_pairs", "n_ls_pairs", "n_mid_pairs"]].mean(numeric_only=True)

,n_annotations,n_ss_pairs,n_ls_pairs,n_mid_pairs
file_label,,,,
clean,0.0,8.0,48.0,0.0
hrf_100,34.5,8.0,48.0,0.0
hrf_20,34.5,8.0,48.0,0.0
hrf_50,34.5,8.0,48.0,0.0


In [63]:
all_event_summary_df.groupby(["file_label", "description"]).size().reset_index(name="count")

,file_label,description,count
0,hrf_100,1,483
1,hrf_20,1,483
2,hrf_50,1,483


# Rank long-separation HbO channels by event-locked effect size

In [64]:
subject_name = "Subj100"
subject_dir = rs_data_dir / subject_name

file_label = "hrf_100"
file_name = "resting_hrf_100.snirf"
file_path = subject_dir / file_name

raw_cw = mne.io.read_raw_snirf(file_path, preload=True, verbose=False)
raw_od = optical_density(raw_cw.copy())
raw_hb = beer_lambert_law(raw_od, ppf=0.1)

events, event_id = mne.events_from_annotations(raw_hb, verbose=False)

epochs_hb = mne.Epochs(
    raw_hb,
    events=events,
    event_id=event_id,
    tmin=-5.0,
    tmax=20.0,
    baseline=None,
    preload=True,
    detrend=None,
    verbose=False,
)

print(epochs_hb)
print(event_id)

<Epochs |  35 events (all good), -5 - 20 sec, baseline off, ~37.5 MB, data loaded,
 '1': 35>
{'1': 1}


/tmp/ipykernel_384858/2417804758.py:8: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.



In [65]:
def build_hb_channel_table(raw_hb, subject_name, file_label):
    picks_hbo = mne.pick_types(raw_hb.info, fnirs="hbo")
    picks_hbr = mne.pick_types(raw_hb.info, fnirs="hbr")
    picks_hb = np.sort(np.concatenate([picks_hbo, picks_hbr]))

    hb_names = np.array(raw_hb.ch_names)[picks_hb]
    hb_types = np.array(raw_hb.get_channel_types())[picks_hb]
    pair_names = np.array([channel_name.split(" ")[0] for channel_name in hb_names])

    distances_all = source_detector_distances(raw_hb.info)
    distances_hb = distances_all[picks_hb]

    ss_mask_all = short_channels(raw_hb.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_hb]
    ls_mask = distances_hb >= 0.025

    hb_channel_table = pd.DataFrame({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": hb_names,
        "pair_name": pair_names,
        "chromophore": hb_types,
        "distance_m": distances_hb,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    hb_channel_table["group"] = np.select(
        [hb_channel_table["is_ss"], hb_channel_table["is_ls"]],
        ["SS", "LS"],
        default="MID",
    )

    return hb_channel_table

In [66]:
hb_channel_table = build_hb_channel_table(raw_hb, subject_name, file_label)

ls_hbo_channel_names = hb_channel_table.loc[
    (hb_channel_table["group"] == "LS") & (hb_channel_table["chromophore"] == "hbo"),
    "channel_name",
].tolist()

print(f"Number of LS HbO channels: {len(ls_hbo_channel_names)}")
print(ls_hbo_channel_names[:10])

Number of LS HbO channels: 48
['S1_D1 hbo', 'S1_D7 hbo', 'S2_D1 hbo', 'S2_D2 hbo', 'S2_D7 hbo', 'S2_D8 hbo', 'S3_D2 hbo', 'S3_D3 hbo', 'S3_D8 hbo', 'S3_D9 hbo']


In [67]:
channel_effect_rows = []

baseline_time_mask = (epochs_hb.times >= -2.0) & (epochs_hb.times <= 0.0)
response_time_mask = (epochs_hb.times >= 4.0) & (epochs_hb.times <= 8.0)

for channel_name in ls_hbo_channel_names:
    single_channel_epochs = epochs_hb.copy().pick([channel_name])
    single_channel_data = single_channel_epochs.get_data()[:, 0, :]

    baseline_values = single_channel_data[:, baseline_time_mask].mean(axis=1)
    response_values = single_channel_data[:, response_time_mask].mean(axis=1)

    effect_values = response_values - baseline_values

    channel_effect_rows.append({
        "subject": subject_name,
        "file_label": file_label,
        "channel_name": channel_name,
        "pair_name": channel_name.split(" ")[0],
        "mean_baseline": baseline_values.mean(),
        "mean_response": response_values.mean(),
        "mean_effect_size": effect_values.mean(),
        "std_effect_size": effect_values.std(),
    })

channel_effect_df = pd.DataFrame(channel_effect_rows)
channel_effect_df = channel_effect_df.sort_values("mean_effect_size", ascending=False).reset_index(drop=True)

channel_effect_df.head(15)

,subject,file_label,channel_name,pair_name,mean_baseline,mean_response,mean_effect_size,std_effect_size
0,Subj100,hrf_100,S2_D8 hbo,S2_D8,-0.000015,0.000026,0.000042,0.000046
1,Subj100,hrf_100,S10_D18 hbo,S10_D18,-0.000014,0.000027,0.000040,0.000036
2,Subj100,hrf_100,S11_D25 hbo,S11_D25,-0.000022,0.000017,0.000039,0.000034
3,Subj100,hrf_100,S2_D2 hbo,S2_D2,-0.000011,0.000023,0.000034,0.000046
4,Subj100,hrf_100,S15_D28 hbo,S15_D28,-0.000013,0.000018,0.000031,0.000032
5,Subj100,hrf_100,S16_D28 hbo,S16_D28,-0.000010,0.000020,0.000031,0.000031
6,Subj100,hrf_100,S2_D1 hbo,S2_D1,-0.000010,0.000020,0.000030,0.000054
7,Subj100,hrf_100,S16_D32 hbo,S16_D32,-0.000007,0.000022,0.000028,0.000050
8,Subj100,hrf_100,S6_D11 hbo,S6_D11,-0.000017,0.000011,0.000028,0.000030
9,Subj100,hrf_100,S13_D20 hbo,S13_D20,-0.000016,0.000012,0.000028,0.000060


In [68]:
top_n_channels = 15

channel_rank_figure = px.bar(
    channel_effect_df.head(top_n_channels),
    x="channel_name",
    y="mean_effect_size",
    title=f"{subject_name} {file_label}: top LS HbO channels by event-locked effect size",
)

channel_rank_figure.update_layout(
    width=1100,
    height=500,
    xaxis_tickangle=-45,
)

channel_rank_figure.show()

In [69]:
top_channel_names = channel_effect_df.head(6)["channel_name"].tolist()
top_pair_names = channel_effect_df.head(6)["pair_name"].tolist()

print("Top channel names:")
print(top_channel_names)

print("\nTop pair names:")
print(top_pair_names)

Top channel names:
['S2_D8 hbo', 'S10_D18 hbo', 'S11_D25 hbo', 'S2_D2 hbo', 'S15_D28 hbo', 'S16_D28 hbo']

Top pair names:
['S2_D8', 'S10_D18', 'S11_D25', 'S2_D2', 'S15_D28', 'S16_D28']


In [70]:
top_channel_plot_rows = []

for channel_name in top_channel_names:
    single_channel_epochs = epochs_hb.copy().pick([channel_name])
    single_channel_data = single_channel_epochs.get_data()[:, 0, :]

    mean_time_course = single_channel_data.mean(axis=0)

    top_channel_plot_rows.append(pd.DataFrame({
        "time_s": epochs_hb.times,
        "signal": mean_time_course,
        "channel_name": channel_name,
    }))

top_channel_plot_df = pd.concat(top_channel_plot_rows, ignore_index=True)
top_channel_plot_df.head()

,time_s,signal,channel_name
0,-5.00,-0.000013,S2_D8 hbo
1,-4.98,-0.000013,S2_D8 hbo
2,-4.96,-0.000014,S2_D8 hbo
3,-4.94,-0.000015,S2_D8 hbo
4,-4.92,-0.000015,S2_D8 hbo


In [71]:
top_channel_time_course_figure = px.line(
    top_channel_plot_df,
    x="time_s",
    y="signal",
    color="channel_name",
    title=f"{subject_name} {file_label}: top LS HbO channel time courses",
)

top_channel_time_course_figure.update_layout(
    width=1100,
    height=550,
)

top_channel_time_course_figure.add_vline(x=0.0)
top_channel_time_course_figure.show()

In [72]:
top_channel_amplitude_rows = []

for compare_file_label, compare_file_name in [
    ("hrf_20", "resting_hrf_20.snirf"),
    ("hrf_50", "resting_hrf_50.snirf"),
    ("hrf_100", "resting_hrf_100.snirf"),
]:
    compare_file_path = subject_dir / compare_file_name

    raw_cw_compare = mne.io.read_raw_snirf(compare_file_path, preload=True, verbose=False)
    raw_od_compare = optical_density(raw_cw_compare.copy())
    raw_hb_compare = beer_lambert_law(raw_od_compare, ppf=0.1)

    compare_events, compare_event_id = mne.events_from_annotations(raw_hb_compare, verbose=False)

    compare_epochs = mne.Epochs(
        raw_hb_compare,
        events=compare_events,
        event_id=compare_event_id,
        tmin=-5.0,
        tmax=20.0,
        baseline=(-2.0, 0.0),
        preload=True,
        detrend=None,
        verbose=False,
    )

    available_top_channels = [channel_name for channel_name in top_channel_names if channel_name in compare_epochs.ch_names]

    compare_data = compare_epochs.copy().pick(available_top_channels).get_data()
    mean_compare_time_course = compare_data.mean(axis=(0, 1))

    top_channel_amplitude_rows.append(pd.DataFrame({
        "time_s": compare_epochs.times,
        "signal": mean_compare_time_course,
        "file_label": compare_file_label,
    }))

top_channel_amplitude_df = pd.concat(top_channel_amplitude_rows, ignore_index=True)
top_channel_amplitude_df.head()

/tmp/ipykernel_384858/1023561748.py:10: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/1023561748.py:10: RuntimeWarning:

The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.

/tmp/ipykernel_384858/1023561748.py:10: RuntimeWarning:

The dat

,time_s,signal,file_label
0,-5.00,-0.000006,hrf_20
1,-4.98,-0.000004,hrf_20
2,-4.96,-0.000005,hrf_20
3,-4.94,-0.000006,hrf_20
4,-4.92,-0.000006,hrf_20


In [73]:
top_channel_amplitude_figure = px.line(
    top_channel_amplitude_df,
    x="time_s",
    y="signal",
    color="file_label",
    title=f"{subject_name}: top-channel HbO response across amplitudes",
)

top_channel_amplitude_figure.update_layout(
    width=1100,
    height=500,
)

top_channel_amplitude_figure.add_vline(x=0.0)
top_channel_amplitude_figure.show()